# Stats DPE aout 2026

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [5]:
usecols = ['numero_dpe', 'date_visite_diagnostiqueur', 'identifiant_ban', 'code_postal_ban', 'code_postal_brut', 'id_rnb', 'provenance_id_rnb']
# date columns must not appear in dtype: they are handled by parse_dates
dtype={'numero_dpe': 'string', 'identifiant_ban': 'string', 'code_postal_ban': 'string', 'code_postal_brut': 'string', 'id_rnb': 'string', 'provenance_id_rnb': 'string'}
parse_dates = ['date_visite_diagnostiqueur']
# giving the exact format avoids pandas guessing it row by row (much faster on big files)
date_format = '%Y-%m-%d'

# usecols = ['numero_dpe', 'id_rnb', 'provenance_id_rnb']
# dtype={'numero_dpe': 'string', 'id_rnb': 'string', 'provenance_id_rnb': 'string'}
# parse_dates = []

In [6]:
df_tertiaire = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe01tertiaire.csv', sep=',', usecols=usecols, dtype=dtype, parse_dates=parse_dates, date_format=date_format)
df_tertiaire['file'] = 'tertiaire'

In [7]:
df_existant = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe03existant.csv', sep=',', usecols=usecols, dtype=dtype, parse_dates=parse_dates, date_format=date_format)
df_existant['file'] = 'existant'

In [8]:
df_neuf = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe02neuf.csv', sep=',', usecols=usecols, dtype=dtype, parse_dates=parse_dates, date_format=date_format)
df_neuf['file'] = 'neuf'

In [9]:
df = pd.concat([df_tertiaire, df_neuf, df_existant])

In [11]:
df.groupby(['provenance_id_rnb']).count()

,numero_dpe,date_visite_diagnostiqueur,id_rnb,code_postal_ban,identifiant_ban,code_postal_brut,file
provenance_id_rnb,,,,,,,
Logiciel,787181,787181,787181,782793,782793,787181,787181
Reprise RNB,7658539,7658539,7658539,7658539,7658539,7658539,7658539


In [14]:
# filtre les DPE dont la visite a eu lieu après le 31/7/2025
df_recent = df[df['date_visite_diagnostiqueur'] > '2025-07-31']
# dropna=False keeps the rows without provenance_id_rnb as their own group
df_recent.groupby(['provenance_id_rnb'], dropna=False).size()

provenance_id_rnb
Logiciel        738368
Reprise RNB         31
<NA>           2185393
dtype: int64

In [16]:
# part des DPE dont l'ID RNB vient du logiciel, mois par mois depuis juillet 2025
df_mensuel = df[df['date_visite_diagnostiqueur'] >= '2025-07-01'].copy()
df_mensuel['mois'] = df_mensuel['date_visite_diagnostiqueur'].dt.to_period('M')
# fillna(False) : les DPE sans provenance comptent dans le total, pas dans les "Logiciel"
df_mensuel['is_logiciel'] = df_mensuel['provenance_id_rnb'].eq('Logiciel').fillna(False)

stats_mensuelles = df_mensuel.groupby('mois').agg(
    total=('numero_dpe', 'size'),
    logiciel=('is_logiciel', 'sum'),
)
stats_mensuelles['pct_logiciel'] = (100 * stats_mensuelles['logiciel'] / stats_mensuelles['total']).round(1)
stats_mensuelles

,total,logiciel,pct_logiciel
mois,,,
2025-07,299567,11217,3.7
2025-08,216367,10637,4.9
2025-09,295169,40535,13.7
2025-10,305900,78796,25.8
2025-11,255974,64609,25.2
2025-12,231516,60006,25.9
2026-01,254745,70413,27.6
2026-02,249451,72835,29.2
2026-03,269753,78200,29.0
